In [1]:
import os
import sys
from langchain_openai import ChatOpenAI
# Adds the parent directory of 'src' to the path
sys.path.append(os.path.abspath(os.path.join("..", "..")))
import mlflow
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from mlflow.genai.optimize import GepaPromptOptimizer
from mlflow.genai.scorers import Equivalence
from rich import print

from src.chatbot.graphs.graph import search_zalando_faq
from src.chatbot.schema.basemodel import QueryExpansion
# from src.prompt_management.models import new_llm

mlflow.tracing.disable_notebook_display()

/home/leonelbaptista/Projects/mrig/chatbot-eval/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
faq_eval_data = mlflow.genai.get_dataset(dataset_id="d-590c5b50328843df8f5fb774a200496e")
faq_prompt = mlflow.genai.load_prompt(name_or_uri="prompts:/faq_prompt@dev")

In [3]:
from langchain.agents import create_agent
from langchain_core.tools import tool

from src.chatbot.agents.models import llm
from src.chatbot.prompts.prompts import multi_query_prompt
from src.chatbot.rag.retrievers.retriever import get_chunks, get_docs
from src.chatbot.schema.basemodel import QueryExpansion


def get_multi_queries(query: str) -> QueryExpansion:
    """
    Process a User Question into three specific formats.

    :param query: The user query to process.
    :return: An object containing the original query and the processed queries.
    """
    prompt = multi_query_prompt.format(query=query)
    structured_llm = llm.with_structured_output(QueryExpansion)
    result = structured_llm.invoke(prompt)
    return result

@tool
def search_zalando_faq(query: str) -> str:
    """
    Search Zalando's FAQ documents with the given user query.

    Args:
        query (str): The user query to search for.

    Returns:
        str: Context string containing results and language instructions.
    """
    expansion = get_multi_queries(query)
    search_queries = getattr(expansion, "queries", [query])
    target_lang = getattr(expansion, "detected_language", "the user's language")

    results = [get_docs(get_chunks(q)) for q in search_queries]

    unique_docs = sorted(set().union(*results), key=lambda x: x[0])

    if not unique_docs:
        return f"CRITICAL: Respond in {target_lang}.\n\nKNOWLEDGE BASE CONTEXT:\nNo context available."

    merged_parts = []
    for i, (page, content) in enumerate(unique_docs):
        if i == 0:
            merged_parts.append(content)
        else:
            prev_page = unique_docs[i - 1][0]
            separator = "\n" if page == prev_page + 1 else "\n\n" 
            merged_parts.append(separator + content)

    context_str = "".join(merged_parts)

    return (
        f"CRITICAL: The user is speaking {target_lang}. "
        f"You MUST respond in {target_lang}.\n\n"
        f"KNOWLEDGE BASE CONTEXT:\n{context_str}"
    )


In [4]:
import json

with open("/home/leonelbaptista/Projects/mrig/chatbot-eval/test_data/evaluation_dataset.json", "r") as file:
    data = json.load(file)

faq_eval_data = data.get("faq_prompt_dataset")

In [6]:
@mlflow.trace
def predict_fn_base_model(question: str) -> str:
    query = question
    agent = create_agent(
        llm, 
        system_prompt=faq_prompt.format(), 
        tools=[search_zalando_faq])
    result = agent.invoke({"messages": [HumanMessage(query)]})
    messages = result.get("messages", [])
    final_answer = messages[-1].content if messages else ""

    return final_answer


In [7]:
response = predict_fn_base_model("What payment methods are accepted on Zalando?")

In [8]:
print(response)

Zalando accepts the following payment methods:

- Credit Cards: Mastercard, Visa, American Express, Diners Club, Discover. Note that a pre-authorisation may appear
on your card but the charge occurs only when the order is dispatched.
- PayPal: You will be redirected to PayPal at checkout. Refunds go back to your PayPal account or bank account if 
the PayPal account is closed.
- Prepayment (Bank Transfer): After ordering, you receive bank transfer details. Items are reserved for 7 days 
until payment is received, and delivery starts after payment is processed.
- Invoice: Pay only for the items you keep. You can pay via PayPal, credit card, or bank transfer for kept items 
after receiving an email 4 days after shipping.
- SEPA Direct Debit: Provide account holder name and IBAN. The direct debit authorisation lasts 36 months, and 
payment is debited once the order is shipped.
- Klarna: Split purchase into 3 interest-free payments. Available for orders between 25,00 € and 5000,00 €. Billing
and shipping countries must match.
- Apple Pay: Available on supported devices and browsers, using Face ID, Touch ID, or passcode.

All payment transactions are secured with certified SSL encryption and 3DS authentication for credit cards.

If you need more details about any method, feel free to ask!

In [11]:
import mlflow
import openai
from mlflow.genai.optimize import GepaPromptOptimizer
from mlflow.genai.datasets import create_dataset
from mlflow.genai.scorers import Equivalence

In [9]:
with mlflow.start_run() as run:
    for record in faq_eval_data:
        predict_fn_base_model(**record["inputs"])

🏃 View run likeable-hound-980 at: http://localhost:5000/#/experiments/1/runs/de86aab0dcc34b63b2d4dff84d7fc493
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [12]:
# Create dataset
dataset = create_dataset(name="faq_dataset")

# Retrieve traces from the run
traces = mlflow.search_traces(return_type="list", run_id=run.info.run_id)

# Merge traces into dataset
dataset.merge_records(traces)

In [3]:
import os
import mlflow
from dotenv import load_dotenv
# Import the actual tracking store backend
from mlflow.store.tracking.sqlalchemy_store import SqlAlchemyStore

load_dotenv()

uri = os.getenv("MLFLOW_TRACKING_URI") # e.g., "sqlite:///mlflow.db"

if uri.startswith("sqlite"):
    # Initialize the store directly with your local DB URI
    store = SqlAlchemyStore(uri)
    
    # This is the actual backend method you were looking for
    store.hard_delete_experiment_metadata()
    print(f"Permanently purged deleted experiments from {uri}")
else:
    print("Hard delete metadata is only supported on direct SQL backend stores.")

Hard delete metadata is only supported on direct SQL backend stores.
